# 2 - Exploratory Data Analysis (EDA)

## Hotel Booking Cancellation Prediction

This notebook is prepared for the EDA stage of the IT3051 FDM Mini Project. The current completed section is **Member 2 - Stay & Reservation Analysis**. Placeholder sections are included for Member 1 and Member 3 so their analyses can be added later.

## EDA Objective

The purpose of EDA is to explore the Hotel Booking dataset to identify cancellation-related patterns, feature distributions, unusual observations, and potential feature-engineering opportunities. Findings from this stage will guide preprocessing, feature engineering, feature selection, and future modelling decisions.

## Import Libraries and Load Dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 120)
sns.set_theme(style='whitegrid', palette='Set2')

DATA_PATH = Path('../data/raw/hotel_booking.csv')
FIGURE_DIR = Path('Figures/stay_reservation')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

## Helper Functions

In [ ]:
def cancellation_rate_table(data, column):
    summary = (
        data.groupby(column, dropna=False)['is_canceled']
        .agg(total_bookings='count', canceled_bookings='sum', cancellation_rate='mean')
        .reset_index()
    )
    summary['cancellation_rate'] = (summary['cancellation_rate'] * 100).round(2)
    return summary.sort_values('cancellation_rate', ascending=False)

def save_current_figure(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved figure: {path}')

## Overall Target Distribution

This gives context for Member 2 target comparisons. A cancellation rate of the full dataset is useful as a baseline.

In [ ]:
target_summary = df['is_canceled'].value_counts().sort_index().reset_index()
target_summary.columns = ['is_canceled', 'count']
target_summary['meaning'] = target_summary['is_canceled'].map({0: 'Not canceled', 1: 'Canceled'})
target_summary['percentage'] = (target_summary['count'] / len(df) * 100).round(2)
target_summary

## Member 1 - Booking & Customer Behaviour Analysis (Placeholder)

This section is reserved for Member 1. Suggested variables include lead time, previous cancellations, previous bookings not canceled, repeated guest, booking changes, days in waiting list, special requests, customer type, and related target comparisons.

## Member 2 - Stay & Reservation Analysis

This section analyses stay and reservation attributes assigned to Member 2:

- Hotel type
- Arrival period
- Weekend-night stays
- Week-night stays
- Adults, children, babies
- Meal
- Reserved room type
- Parking requirements
- Related target comparisons
- Outliers or unusual observations
- Feature-engineering opportunities such as `total_stay` and `total_guests`

### 2.1 Hotel Type Distribution and Cancellation Rate

In [ ]:
hotel_counts = df['hotel'].value_counts().reset_index()
hotel_counts.columns = ['hotel', 'count']
hotel_counts['percentage'] = (hotel_counts['count'] / len(df) * 100).round(2)
hotel_counts

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='hotel', hue='is_canceled')
plt.title('Hotel Type by Cancellation Status')
plt.xlabel('Hotel Type')
plt.ylabel('Number of Bookings')
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
save_current_figure('hotel_type_by_cancellation.png')

In [ ]:
cancellation_rate_table(df, 'hotel')

**Finding:** Hotel type is relevant because City Hotel and Resort Hotel bookings may have different cancellation behavior. This variable should be retained as a categorical predictor and encoded during preprocessing.

### 2.2 Arrival Period Analysis

Arrival month, year, week number, and day of month help reveal seasonality and period-based booking patterns.

In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
arrival_month_summary = cancellation_rate_table(df, 'arrival_date_month')
arrival_month_summary['arrival_date_month'] = pd.Categorical(arrival_month_summary['arrival_date_month'], categories=month_order, ordered=True)
arrival_month_summary.sort_values('arrival_date_month')

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df, x='arrival_date_month', hue='is_canceled', order=month_order)
plt.title('Arrival Month by Cancellation Status')
plt.xlabel('Arrival Month')
plt.ylabel('Number of Bookings')
plt.xticks(rotation=45)
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
save_current_figure('arrival_month_by_cancellation.png')

In [ ]:
arrival_year_summary = cancellation_rate_table(df, 'arrival_date_year')
arrival_year_summary

**Finding:** Arrival-period variables can capture seasonality. During preprocessing/feature engineering, month and year should be handled as time-related categorical or ordinal features depending on the modelling approach.

### 2.3 Stay Duration Analysis

This section examines weekend-night stays and week-night stays. It also proposes `total_stay` as a future engineered feature.

In [ ]:
stay_columns = ['stays_in_weekend_nights', 'stays_in_week_nights']
df[stay_columns].describe().T.round(2)

In [ ]:
plt.figure(figsize=(12, 4))
for i, column in enumerate(stay_columns, start=1):
    plt.subplot(1, 2, i)
    sns.boxplot(data=df, x='is_canceled', y=column)
    plt.title(f'{column} by Cancellation Status')
    plt.xlabel('is_canceled')
    plt.ylabel(column)
save_current_figure('stay_nights_boxplots.png')

In [ ]:
df_member2 = df.copy()
df_member2['total_stay'] = df_member2['stays_in_weekend_nights'] + df_member2['stays_in_week_nights']
total_stay_summary = df_member2['total_stay'].describe().to_frame('total_stay').T.round(2)
zero_night_count = int((df_member2['total_stay'] == 0).sum())
total_stay_summary['zero_night_bookings'] = zero_night_count
total_stay_summary

In [ ]:
plt.figure(figsize=(9, 4))
sns.histplot(data=df_member2, x='total_stay', hue='is_canceled', bins=40, kde=False, multiple='stack')
plt.title('Total Stay Distribution by Cancellation Status')
plt.xlabel('Total Stay Nights')
plt.ylabel('Number of Bookings')
save_current_figure('total_stay_distribution.png')

**Finding:** Weekend and week-night stays can be combined into `total_stay`. Zero-night and very long-stay records should be investigated before preprocessing decisions are made.

### 2.4 Guest Composition Analysis

This section analyses adults, children, and babies. It also proposes `total_guests` as a possible engineered feature.

In [ ]:
guest_columns = ['adults', 'children', 'babies']
df[guest_columns].describe().T.round(2)

In [ ]:
df_member2['children'] = df_member2['children'].fillna(0)
df_member2['total_guests'] = df_member2['adults'] + df_member2['children'] + df_member2['babies']
guest_summary = df_member2[['adults', 'children', 'babies', 'total_guests']].describe().T.round(2)
guest_summary['zero_guest_bookings'] = [
    int((df_member2['adults'] == 0).sum()),
    int((df_member2['children'] == 0).sum()),
    int((df_member2['babies'] == 0).sum()),
    int((df_member2['total_guests'] == 0).sum())
]
guest_summary

In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(data=df_member2[df_member2['total_guests'] <= 6], x='total_guests', hue='is_canceled')
plt.title('Total Guests by Cancellation Status')
plt.xlabel('Total Guests')
plt.ylabel('Number of Bookings')
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
save_current_figure('total_guests_by_cancellation.png')

**Finding:** `total_guests` is a useful feature-engineering opportunity. Records with zero total guests should be treated as unusual and investigated during preprocessing.

### 2.5 Meal Type Analysis

In [ ]:
df['meal'].value_counts(dropna=False).to_frame('count')

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='meal', hue='is_canceled')
plt.title('Meal Type by Cancellation Status')
plt.xlabel('Meal Type')
plt.ylabel('Number of Bookings')
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
save_current_figure('meal_type_by_cancellation.png')

In [ ]:
cancellation_rate_table(df, 'meal')

**Finding:** Meal is a categorical feature and should be encoded during preprocessing. Undefined or rare categories should be reviewed before encoding.

### 2.6 Reserved Room Type Analysis

In [ ]:
df['reserved_room_type'].value_counts(dropna=False).to_frame('count')

In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=df, x='reserved_room_type', hue='is_canceled', order=sorted(df['reserved_room_type'].unique()))
plt.title('Reserved Room Type by Cancellation Status')
plt.xlabel('Reserved Room Type')
plt.ylabel('Number of Bookings')
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
save_current_figure('reserved_room_type_by_cancellation.png')

In [ ]:
cancellation_rate_table(df, 'reserved_room_type')

**Finding:** Reserved room type may reflect customer preference and booking conditions. It should be retained as a categorical feature, while rare room categories may require careful encoding.

### 2.7 Parking Requirement Analysis

In [ ]:
df['required_car_parking_spaces'].value_counts(dropna=False).sort_index().to_frame('count')

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='required_car_parking_spaces', hue='is_canceled')
plt.title('Required Parking Spaces by Cancellation Status')
plt.xlabel('Required Car Parking Spaces')
plt.ylabel('Number of Bookings')
plt.legend(title='is_canceled', labels=['Not canceled', 'Canceled'])
save_current_figure('parking_spaces_by_cancellation.png')

In [ ]:
cancellation_rate_table(df, 'required_car_parking_spaces')

**Finding:** Parking requests may indicate stronger booking intention. This variable is numerical/discrete and may be useful without complex transformation, but rare high values should be reviewed.

### 2.8 Member 2 Outlier and Unusual Observation Summary

In [ ]:
member2_unusual = pd.DataFrame([
    {'check': 'Zero total stay nights', 'count': int((df_member2['total_stay'] == 0).sum())},
    {'check': 'Very long stays above 30 nights', 'count': int((df_member2['total_stay'] > 30).sum())},
    {'check': 'Missing children values', 'count': int(df['children'].isna().sum())},
    {'check': 'Zero total guests', 'count': int((df_member2['total_guests'] == 0).sum())},
    {'check': 'Bookings with more than 6 guests', 'count': int((df_member2['total_guests'] > 6).sum())},
    {'check': 'Undefined meal values', 'count': int((df['meal'] == 'Undefined').sum())},
    {'check': 'Parking spaces greater than 2', 'count': int((df['required_car_parking_spaces'] > 2).sum())},
])
member2_unusual

### 2.9 Member 2 Preprocessing and Feature-Engineering Implications

- Encode categorical variables: `hotel`, `arrival_date_month`, `meal`, and `reserved_room_type`.
- Consider engineering `total_stay = stays_in_weekend_nights + stays_in_week_nights`.
- Consider engineering `total_guests = adults + children + babies`.
- Investigate zero-night bookings and zero-guest bookings before cleaning.
- Handle the small number of missing `children` values during preprocessing.
- Review rare categories in `meal` and `reserved_room_type` before encoding.
- Keep `required_car_parking_spaces` as a useful discrete numerical feature, while reviewing rare high values.

## Member 3 - Commercial & Distribution Analysis (Placeholder)

This section is reserved for Member 3. Suggested variables include market segment, distribution channel, deposit type, agent/company, ADR, country, assigned room type, correlations, high-cardinality variables, missing-value patterns, and data-leakage variables such as `reservation_status` and `reservation_status_date`.

## Overall EDA Conclusion and Recommendations (To Be Completed After All Members Finish)

This section should combine Member 1, Member 2, and Member 3 findings into one final EDA conclusion and preprocessing/feature-engineering recommendation section.